# Saw - Simple Additive Weighting

In [1]:
using Pkg; Pkg.activate(".")

  Activating project at `~/code/julia/notebooks`


In [3]:
using JMcDM, DataFrames

In [4]:
df = DataFrame(
    price             = [2500.0, 3000, 4000, 5000],
    rooms             = [2.0   ,  2,   3,    3   ],
    size              = [50.0  , 75,  100, 100   ],
    distance_to_transportation = [1000.0, 1200, 800, 750  ],
    years             = [3.0,  4,  8,  7]
)

Row,price,rooms,size,distance_to_transportation,years
,Float64,Float64,Float64,Float64,Float64
1,2500.0,2.0,50.0,1000.0,3.0
2,3000.0,2.0,75.0,1200.0,4.0
3,4000.0,3.0,100.0,800.0,8.0
4,5000.0,3.0,100.0,750.0,7.0


In [5]:
w = [0.40, 0.35, 0.10, 0.10, 0.05];

In [6]:
directions = [minimum, maximum, maximum, minimum, minimum];

In [7]:
sawresult = JMcDM.saw(Matrix(df), w, directions);

In [8]:
sawresult.scores

4-element Vector{Float64}:
 0.8083333333333333
 0.7416666666666666
 0.8125
 0.7714285714285715

In [23]:
sawresult.bestIndex

3

In [24]:
sawresult.ranking

4-element Vector{Int64}:
 3
 1
 4
 2

## Manual Calculations 

In [16]:
### Normalized Matrix (divideByColumnMinMax)

In [15]:
begin 
    mat = Matrix(df)
    normalized = copy(mat)
    normalized[:, 1] = minimum(normalized[:, 1]) ./ normalized[:, 1]
    normalized[:, 2] = normalized[:, 2] ./ maximum(normalized[:, 2])
    normalized[:, 3] = normalized[:, 3] ./ maximum(normalized[:, 3])
    normalized[:, 4] = minimum(normalized[:, 4]) ./ normalized[:, 4]
    normalized[:, 5] = minimum(normalized[:, 5]) ./ normalized[:, 5]
    display(normalized)
end

4×5 Matrix{Float64}:
 1.0       0.666667  0.5   0.75    1.0
 0.833333  0.666667  0.75  0.625   0.75
 0.625     1.0       1.0   0.9375  0.375
 0.5       1.0       1.0   1.0     0.428571

In [17]:
### Weighted Normalized Matrix

In [19]:
begin
    wnormalized = copy(normalized)
    for i in 1:5
        wnormalized[:, i] = wnormalized[:, i] * w[i]
    end
    display(wnormalized)
end

4×5 Matrix{Float64}:
 0.4       0.233333  0.05   0.075    0.05
 0.333333  0.233333  0.075  0.0625   0.0375
 0.25      0.35      0.1    0.09375  0.01875
 0.2       0.35      0.1    0.1      0.0214286

In [20]:
### Row sums

In [33]:
rowsums = sum(wnormalized, dims = 2)

4×1 Matrix{Float64}:
 0.8083333333333333
 0.7416666666666666
 0.8125
 0.7714285714285715

In [25]:
### Ranking

In [37]:
sortperm(rowsums, dims = 1, rev = true)

4×1 Matrix{Int64}:
 3
 1
 4
 2